In [1]:
!git clone https://github.com/JayRothenberger/Meta-Co-Training

fatal: destination path 'Meta-Co-Training' already exists and is not an empty directory.


In [3]:
pip install -e Meta-Co-Training/meta-cotraining


Obtaining file:///content/Meta-Co-Training/meta-cotraining
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Using cached huggingface_hub-0.23.3-py3-none-any.whl.metadata (12 kB)
  Using cached numpy-1.26.4.tar.gz (15.8 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pandas-2.2.2.tar.gz (4.4 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (

In [4]:
!pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-r8n7u7x8
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-r8n7u7x8
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [5]:
# standard library imports
import argparse
from copy import deepcopy as copy
import os
import sys

# Add the cloned repository directory to Python's system path so mct can be imported
repo_path = os.path.abspath('Meta-Co-Training/meta-cotraining/src')
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# installed imports
import torch
import torchvision
import wandb
# local code imports
from mct.MCT import MetaCoTrainingModel
from mct.image_models import IMAGE_DISTANCES, IMAGE_TRANSFORMS
from mct.models import MLPProbe
from mct.utils import subset_npercent_dataset

def setup(rank, world_size):
    # Set default master address and port for local rendezvous
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'

    # Use 'gloo' as it works on both CPU/GPU and doesn't require NCCL compiled on CPU environments
    backend = "nccl" if torch.cuda.is_available() else "gloo"
    torch.distributed.init_process_group(backend, rank=rank, world_size=world_size)


def cleanup():
    torch.distributed.destroy_process_group()

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [6]:
import os
import torch

# Explicitly configure environment variables for single-process rendezvous
os.environ['MASTER_ADDR'] = 'localhost'
os.environ['MASTER_PORT'] = '12355'

# connect to the process group with default rank 0 and world size 1 for single-GPU/CPU environment
setup(rank=0, world_size=1)

# specify the device for the current process
if 'RANK' in os.environ:
    device = int(os.environ['RANK']) % torch.cuda.device_count()
else:
    device = 0 if torch.cuda.is_available() else 'cpu'

torch.manual_seed(13)
if torch.cuda.is_available():
    torch.cuda.set_device(device)

views = ['DINOv2', 'CLIP']
# the view that will be trained on this device
if 'RANK' in os.environ:
    view = views[int(os.environ['RANK']) % len(views)]
else:
    view = views[0]

# the index in the views list will correspond to the index in these lists
# these lists will store the dataset objects for each view
trains = []  # training views
unlbls = []  # unlabeled views
vals = []  # validation views

In [7]:
# Define parameters directly since 'args' is not defined
class Args:
    train_size = 0.1  # fraction of dataset to use for training (e.g., 10%)

args = Args()

# For each view we create a dataset object and specify the correct transform
for view in views:
    # Use Flowers102 instead of ImageNet and set download=True for Colab
    dataset = torchvision.datasets.Flowers102(
        root='/content/data',
        split='train',
        transform=IMAGE_TRANSFORMS[view],
        download=True
    )

    val = torchvision.datasets.Flowers102(
        root='/content/data',
        split='val',
        transform=IMAGE_TRANSFORMS[view],
        download=True
    )

    # Dynamically patch Flowers102 to have a .samples attribute compatible with subset_npercent_dataset
    # subset_npercent_dataset expects elements of .samples to be pairs (item, target)
    dataset.samples = list(zip(dataset._image_files, dataset._labels))
    val.samples = list(zip(val._image_files, val._labels))

    # Split the data into labeled and unlabeled pools
    train, unlbl = subset_npercent_dataset(dataset, percent=args.train_size * 100)

    # Append the dataset objects to the view lists
    trains.append(train)
    unlbls.append(unlbl)
    vals.append(val)

# Update the number of classes from 1000 (ImageNet) to 102 (Oxford Flowers)
num_classes = 102

0.1
918 102
0.1
918 102


In [8]:
# for each view, get the correct foundation model
models = [IMAGE_DISTANCES[view]().model for view in views]

# Fall back to 'cpu' if no physical CUDA accelerator is available
run_device = 'cpu' if not torch.cuda.is_available() else device

# send each model to the correct run device
for model in models:
    model.to(run_device)

/usr/local/lib/python3.13/dist-packages/torch/hub.py:275: UserWarning: TORCH_HUB is deprecated, please use env TORCH_HOME instead
  hub_dir = get_dir()
/usr/local/lib/python3.13/dist-packages/torch/hub.py:356: UserWarning: TORCH_HUB is deprecated, please use env TORCH_HOME instead
  hub_dir = get_dir()
Using cache found in ./hub/facebookresearch_dinov2_main
/content/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/content/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/content/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
/usr/local/lib/python3.13/dist-packages/torch/hub.py:878: UserWarning: TORCH_HUB is deprecated, please use env TOR

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_at

In [9]:
import os
import torch
import types

# Configure distributed environment variables for single-process notebook execution
os.environ['WORLD_SIZE'] = '2'
os.environ['RANK'] = '0'

# Mock torch.cuda.device_count to return 1 to bypass checks
if not torch.cuda.is_available():
    if not hasattr(torch.cuda, '_original_device_count'):
        torch.cuda._original_device_count = torch.cuda.device_count
    torch.cuda.device_count = lambda: 1

# Define the missing hyperparameters directly in the notebook
shapes = [1024, 768]
warmup_epochs = 10
epochs = 20
batch_size = 64

# append the MLP head on top of the representation model
models = [MLPProbe(model, shape, num_classes) for shape, model in zip(shapes, models)]

# Monkey-patch 'torch.Tensor.to' and the models' '.to' methods to redirect integer device targets to 'cpu' if CUDA is not available
if not torch.cuda.is_available():
    # Use the root PyTorch implementation to avoid infinite recursion loops on re-runs
    _original_tensor_to = torch._C._TensorBase.to

    def safe_tensor_to(self, *args, **kwargs):
        if args and isinstance(args[0], int):
            args = ('cpu',) + args[1:]
        elif 'device' in kwargs and isinstance(kwargs['device'], int):
            kwargs['device'] = 'cpu'
        return _original_tensor_to(self, *args, **kwargs)

    torch.Tensor.to = safe_tensor_to

    def safe_to(self, *args, **kwargs):
        if args and isinstance(args[0], int):
            args = ('cpu',) + args[1:]
        elif 'device' in kwargs and isinstance(kwargs['device'], int):
            kwargs['device'] = 'cpu'
        return self._original_to(*args, **kwargs)

    for m in models:
        if not hasattr(m, '_original_to'):
            m._original_to = m.to
            m.to = types.MethodType(safe_to, m)

# Extract raw datasets from the DataLoaders recursively if they are wrapped
trains_dataset = [getattr(t, 'dataset', t) for t in trains]
unlbls_dataset = [getattr(u, 'dataset', u) for u in unlbls]
vals_dataset = [getattr(v, 'dataset', v) for v in vals]

# Ensure we unwrap recursively if multiple DataLoaders are nested
trains_dataset = [getattr(t, 'dataset', t) for t in trains_dataset]
unlbls_dataset = [getattr(u, 'dataset', u) for u in unlbls_dataset]
vals_dataset = [getattr(v, 'dataset', v) for v in vals_dataset]

# construct the MCT model object which implements a custom training loop
MCTModel = MetaCoTrainingModel(models)

# supervised warmup
states = MCTModel.train(warmup_epochs,
                    warmup_epochs + 1,
                    trains_dataset,
                    unlbls_dataset,
                    vals_dataset,
                    vals_dataset,
                    checkpoint_path=f'./chkpts/{view}_chkpt',
                    batch_size=batch_size,
                    log_interval=100
                    )
# preparation stage in which the model does not alter embedder weights
states = MCTModel.train(epochs,
                    0,
                    trains_dataset,
                    copy(unlbls_dataset),
                    copy(vals_dataset),
                    copy(vals_dataset),
                    checkpoint_path=f'./chkpts/{view}_chkpt',
                    batch_size=batch_size,
                    log_interval=100
                    )

10 0


/content/Meta-Co-Training/meta-cotraining/src/mct/MCT.py:232: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scalers = [GradScaler() for model in self.models]
/usr/local/lib/python3.13/dist-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(
/content/Meta-Co-Training/meta-cotraining/src/mct/MCT.py:234: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  mct_scaler = torch.cuda.amp.GradScaler()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running

10 1


16it [50:32, 189.53s/it]


10 2


...

10 5



16it [50:11, 188.22s/it]

10 6



3it [12:41, 253.70s/it]


TypeError: Caught TypeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
TypeError: 'DataLoader' object is not subscriptable
